# Making list of filenames

In [1]:
import matplotlib.pyplot as plt
from obspy import read_events
from obspy import UTCDateTime
import numpy as np
import pandas as pd
import scipy.stats as stats
import wget
import csv

In [2]:
# Catalog Data
big10_catalog = read_events("Big10_Greece_Seismicity.xml")
filenames = []

j=1
for event in big10_catalog: # For each earthquake

    # Read earthquake data
    origin = event.preferred_origin() or event.origins[0]
    event_time = origin.time
    focal_mech = event.preferred_focal_mechanism() or event.focal_mechanisms[0]
    moment_tensor = focal_mech.moment_tensor
    nodal_planes = focal_mech.nodal_planes
    plane1 = nodal_planes.nodal_plane_1
    plane2 = nodal_planes.nodal_plane_2
    
    mag = event.preferred_magnitude() or event.magnitudes[0]
    mag = float(mag.mag)

    for plane in (plane1,plane2):
        filenames.append(f"Greece_EQ{j}_M{mag}_{plane}_{event_time}.csv")
    j=j+1

    df = pd.DataFrame(filenames)
    df.to_csv("filenames.csv", index=False, header=None)

# Downloading the files

In [3]:
# Catalog Data
filenames = pd.read_csv("filenames.csv", names=['col'], header=None)
filenames=filenames['col'].tolist()
print(filenames)

['Greece_EQ1_M6.72_NodalPlane(strike=201.0, dip=44.0, rake=55.0)_2006-01-08T11:35:00.300000Z.csv', 'Greece_EQ1_M6.72_NodalPlane(strike=66.0, dip=55.0, rake=119.0)_2006-01-08T11:35:00.300000Z.csv', 'Greece_EQ2_M6.85_NodalPlane(strike=332.0, dip=6.0, rake=120.0)_2008-02-14T10:09:29.000000Z.csv', 'Greece_EQ2_M6.85_NodalPlane(strike=121.0, dip=85.0, rake=87.0)_2008-02-14T10:09:29.000000Z.csv', 'Greece_EQ3_M6.54_NodalPlane(strike=337.0, dip=5.0, rake=127.0)_2008-02-14T12:09:02.700000Z.csv', 'Greece_EQ3_M6.54_NodalPlane(strike=120.0, dip=86.0, rake=87.0)_2008-02-14T12:09:02.700000Z.csv', 'Greece_EQ4_M6.76_NodalPlane(strike=339.0, dip=3.0, rake=130.0)_2013-10-12T13:11:56.400000Z.csv', 'Greece_EQ4_M6.76_NodalPlane(strike=119.0, dip=88.0, rake=88.0)_2013-10-12T13:11:56.400000Z.csv', 'Greece_EQ5_M6.86_NodalPlane(strike=73.0, dip=85.0, rake=-177.0)_2014-05-24T09:25:18.800000Z.csv', 'Greece_EQ5_M6.86_NodalPlane(strike=343.0, dip=87.0, rake=-5.0)_2014-05-24T09:25:18.800000Z.csv', 'Greece_EQ6_M6.5_N

In [4]:
EQ=1
for i in range(20):
    filename=filenames[i]
    print(EQ)
    print(f"GNSSVerify/EQ{EQ}.{2-(i+1)%2}/")
    print(filename)
    relevant_stations = pd.read_csv(f"Finite/{filename}")
    sta_ids = relevant_stations["Station_ID"]
    for station in sta_ids:
        url = f"https://geodesy.unr.edu/gps_timeseries/IGS20/tenv3/EU/{station.upper()}.EU.tenv3"
        wget.download(url, out=f"GNSSVerify/EQ{EQ}.{2-(i+1)%2}/")
    if ((i+1)%2==0):
        EQ=EQ+1

1
GNSSVerify/EQ1.1/
Greece_EQ1_M6.72_NodalPlane(strike=201.0, dip=44.0, rake=55.0)_2006-01-08T11:35:00.300000Z.csv
100% [....................................................] 300696 / 3006961
GNSSVerify/EQ1.2/
Greece_EQ1_M6.72_NodalPlane(strike=66.0, dip=55.0, rake=119.0)_2006-01-08T11:35:00.300000Z.csv
100% [....................................................] 300696 / 3006962
GNSSVerify/EQ2.1/
Greece_EQ2_M6.85_NodalPlane(strike=332.0, dip=6.0, rake=120.0)_2008-02-14T10:09:29.000000Z.csv
100% [....................................................] 608736 / 6087362
GNSSVerify/EQ2.2/
Greece_EQ2_M6.85_NodalPlane(strike=121.0, dip=85.0, rake=87.0)_2008-02-14T10:09:29.000000Z.csv
100% [....................................................] 608736 / 6087363
GNSSVerify/EQ3.1/
Greece_EQ3_M6.54_NodalPlane(strike=337.0, dip=5.0, rake=127.0)_2008-02-14T12:09:02.700000Z.csv
100% [....................................................] 608736 / 6087363
GNSSVerify/EQ3.2/
Greece_EQ3_M6.54_NodalPlane(st

# Computing Coseismic Deformation from GNSS time-series

In [1]:
%%bash

t1_list=( # A year before event
    None
    2005.0192 # 2006-01-08
    2007.1202 # 2008-02-14
    2007.1202 # 2008-02-14
    2012.7781 # 2013-10-12
    2013.3918 # 2014-05-24
    2014.8767 # 2015-11-17
    2016.5479 # 2017-07-20
    2017.8137 # 2018-10-25
    2019.3333 # 2020-05-02
    2019.8279 # 2020-10-30
)

t2_list=( # A year after event
    None
    2007.0192 # 2006-01-08
    2009.1202 # 2008-02-14
    2009.1202 # 2008-02-14
    2014.7781 # 2013-10-12
    2015.3918 # 2014-05-24
    2016.8767 # 2015-11-17
    2018.5479 # 2017-07-20
    2019.8137 # 2018-10-25
    2021.3333 # 2020-05-02
    2021.8279 # 2020-10-30
)

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables

for i in $(seq 1 10); do
    t1=${t1_list[$i]}
    t2=${t2_list[$i]}

    for j in $(seq 1 2); do
        dirIN=${dirEXE}/EQ${i}.${j} # Input Folder
        dirOUT=${dirIN} # Output Folder

        cd $dirEXE # Go to executables folder
        files=("$dirIN"/*.EU.tenv3)

        if [ ! -e "${files[0]}" ]; then
            continue
        fi

        # For every .tenv3 file in the folder
        for f in "${files[@]}"; do
            sta=$(basename "$f" .EU.tenv3) # Station Code/ Identifier
            rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run
        
            # Reads file "$f", filters in range [t1,t2], and extracts 3:Decimal Year and 11:North to serv.inp
            gawk -v tin="$t1" -v tout="$t2" -v c=11 'NR>1 && $3>=tin && $3<=tout {print $3, $c*1000}' "$f" > serv.inp
            if [ -s serv.inp ]; then
                octave -q < input_cycleslip.m
                [ -f serv.bayes ] && cp serv.bayes "$dirOUT/$sta.N.bayes.out"
                [ -f serv.p_tau ] && cp serv.p_tau "$dirOUT/$sta.N.bayes.p_tau"
            else
                echo "Warning: No data for $sta in window $t1 - $t2"
            fi
            
            rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run
        
            # Reads file "$f", filters in range [t1,t2], and extracts 3:Decimal Year and 10:East to serv.inp
            gawk -v tin="$t1" -v tout="$t2" -v c=10 'NR>1 && $3>=tin && $3<=tout {print $3, $c*1000}' "$f" > serv.inp
            if [ -s serv.inp ]; then
                octave -q < input_cycleslip.m
                [ -f serv.bayes ] && cp serv.bayes "$dirOUT/$sta.E.bayes.out"
                [ -f serv.p_tau ] && cp serv.p_tau "$dirOUT/$sta.E.bayes.p_tau"
            else
                echo "Warning: No data for $sta in window $t1 - $t2"
            fi
            
            rm -f serv.inp serv.p_tau serv.bayes # Delete residual files from prev run
        
            # Reads file "$f", filters in range [t1,t2], and extracts 3:Decimal Year and 12:Up to serv.inp
            gawk -v tin="$t1" -v tout="$t2" -v c=12 'NR>1 && $3>=tin && $3<=tout {print $3, $c*1000}' "$f" > serv.inp
            if [ -s serv.inp ]; then
                octave -q < input_cycleslip.m
                [ -f serv.bayes ] && cp serv.bayes "$dirOUT/$sta.U.bayes.out"
                [ -f serv.p_tau ] && cp serv.p_tau "$dirOUT/$sta.U.bayes.p_tau"
            else
                echo "Warning: No data for $sta in window $t1 - $t2"
            fi

            rm -f serv.inp serv.p_tau serv.bayes
        done
    done
done

tau0 = 2006.225900000000
tau0 = 2005.169100000000
tau0 = 2005.169100000000
tau0 = 2005.850800000000
tau0 = 2005.166300000000
tau0 = 2005.166300000000
tau0 = 2006.064300000000
tau0 = 2005.401800000000
tau0 = 2005.401800000000
tau0 = 2006.759800000000
tau0 = 2006.518800000000
tau0 = 2006.518800000000
tau0 = 2006.105400000000
tau0 = 2005.048600000000
tau0 = 2005.048600000000
tau0 = 2005.158100000000
tau0 = 2005.048600000000
tau0 = 2005.048600000000
tau0 = 2006.458600000000
tau0 = 2005.048600000000
tau0 = 2005.048600000000
tau0 = 2005.845300000000
tau0 = 2005.388100000000
tau0 = 2005.388100000000
tau0 = 2005.122500000000
tau0 = 2005.048600000000
tau0 = 2005.048600000000
tau0 = 2006.099900000000
tau0 = 2005.048600000000
tau0 = 2005.048600000000
tau0 = 2005.232000000000
tau0 = 2005.048600000000
tau0 = 2005.048600000000
tau0 = 2006.442200000000
tau0 = 2005.048600000000
tau0 = 2005.048600000000
tau0 = 2006.833700000000
tau0 = 2006.603700000000
tau0 = 2006.603700000000
tau0 = 2006.751500000000


error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4
error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4
error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4


tau0 = 2013.642700000000
tau0 = 2012.807700000000
tau0 = 2012.807700000000
tau0 = 2014.414800000000
tau0 = 2012.807700000000
tau0 = 2012.807700000000
tau0 = 2013.601600000000
tau0 = 2012.807700000000
tau0 = 2012.807700000000


error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4
error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4
error: t(2): out of bound 1 (dimensions are 1x1)
error: called from
    posterior_cont_gen at line 36 column 4


tau0 = 2014.694000000000
tau0 = 2013.949300000000
tau0 = 2013.949300000000
tau0 = 2015.071900000000
tau0 = 2015.041800000000
tau0 = 2015.041800000000
tau0 = 2015.014400000000
tau0 = 2013.420900000000
tau0 = 2013.420900000000
tau0 = 2013.645400000000
tau0 = 2013.420900000000
tau0 = 2013.420900000000
tau0 = 2015.203300000000
tau0 = 2013.437400000000
tau0 = 2013.437400000000
tau0 = 2014.313500000000
tau0 = 2013.420900000000
tau0 = 2013.420900000000
tau0 = 2013.749500000000
tau0 = 2013.420900000000
tau0 = 2013.420900000000
tau0 = 2013.752200000000
tau0 = 2013.423700000000
tau0 = 2013.423700000000
tau0 = 2013.749500000000
tau0 = 2013.423700000000
tau0 = 2013.423700000000
tau0 = 2013.574300000000
tau0 = 2013.423700000000
tau0 = 2013.423700000000
tau0 = 2014.833700000000
tau0 = 2014.146500000000
tau0 = 2014.146500000000
tau0 = 2014.395600000000
tau0 = 2013.423700000000
tau0 = 2013.423700000000
tau0 = 2013.574300000000
tau0 = 2013.420900000000
tau0 = 2013.420900000000
tau0 = 2014.433900000000


...

tau0 = 2021.522200000000


    posterior_cont_gen at line 183 column 15

error: 'muk' undefined near line 205, column 7
error: called from
    posterior_cont_gen at line 205 column 6


tau0 = 2021.522200000000


    posterior_cont_gen at line 183 column 15

error: 'muk' undefined near line 205, column 7
error: called from
    posterior_cont_gen at line 205 column 6


tau0 = 2020.339500000000
tau0 = 2020.019200000000
tau0 = 2020.019200000000
tau0 = 2021.787800000000
tau0 = 2021.683800000000
tau0 = 2021.683800000000
tau0 = 2020.829600000000
tau0 = 2020.019200000000
tau0 = 2020.019200000000
tau0 = 2020.846000000000
tau0 = 2019.857600000000
tau0 = 2019.857600000000
tau0 = 2021.388100000000
tau0 = 2020.019200000000
tau0 = 2020.019200000000
tau0 = 2021.388100000000
tau0 = 2020.019200000000
tau0 = 2020.019200000000
tau0 = 2020.832300000000
tau0 = 2020.733700000000
tau0 = 2020.733700000000
tau0 = 2020.840500000000
tau0 = 2020.019200000000
tau0 = 2020.019200000000
tau0 = 2020.637900000000
tau0 = 2019.863100000000
tau0 = 2019.863100000000
tau0 = 2020.826800000000
tau0 = 2020.733700000000
tau0 = 2020.733700000000
tau0 = 2020.632400000000
tau0 = 2020.019200000000
tau0 = 2020.019200000000
tau0 = 2020.509200000000
tau0 = 2020.509200000000
tau0 = 2020.509200000000
tau0 = 2020.646100000000
tau0 = 2019.857600000000
tau0 = 2019.857600000000
tau0 = 2020.796700000000


## Read the displacement corresponding to each earthquake and write to file

In [2]:
%%bash

events=(
    None
    2006.0192 # 2006-01-08
    2008.1202 # 2008-02-14
    2008.1202 # 2008-02-14
    2013.7781 # 2013-10-12
    2014.3918 # 2014-05-24
    2015.8767 # 2015-11-17
    2017.5479 # 2017-07-20
    2018.8137 # 2018-10-25
    2020.3333 # 2020-05-02
    2020.8279 # 2020-10-30
)

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables
disp=(
    "E"
    "N"
    "U"
)

cd ${dirEXE} || exit 1

for i in $(seq 1 10); do # For each quake
    event=${events[$i]}

    for j in $(seq 1 2); do # For each nodal plane
        dirIN=${dirEXE}/EQ${i}.${j} # Input Folder
        dirOUT=${dirIN} # Output Folder
        echo "EQ${i}.${j} : $event" | tee -a GNSSVerify.txt

        for g in ${disp[@]}; do # For each component of displacement
            files=("$dirIN"/*."${g}".bayes.p_tau)

            if [ ! -e "${files[0]}" ]; then
                continue
            fi

            echo ${g} | tee -a GNSSVerify.txt
    
            # For every .p_tau file
            for f in "${files[@]}"; do
                sta=$(basename "$f" ."${g}".bayes.p_tau) # Station Code/ Identifier
                gawk -v sta="$sta" -v val="${event}" -v OFS="\t" '
                    $1 >= val {
                        print sta, $1, $3
                        found = 1
                        exit
                    }
                    END {
                        if (!found) print sta, "NO_DATA_AFTER_EVENT", "N/A"
                    }
                ' "$f" | tee -a GNSSVerify.txt
            done 
        done
    done
done

EQ1.1 : 2006.0192
E
AKYR	2006.17110	0.00000
ANOP	2006.02050	0.00000
ATRS	2006.02050	0.00000
GVDS	2006.48600	0.00000
KERY	2006.02050	0.00000
KITH	2006.02050	0.00000
KOUN	2006.45860	0.00000
KRYO	2006.02050	0.00000
MEN1	2006.02050	0.00000
MET4	2006.02050	0.00000
NEA1	2006.02050	0.00000
PSAR	2006.44220	0.00000
RLSO	2006.57630	0.00000
SPR2	2006.55170	0.00000
TUC2	2006.02050	0.00000
VASS	2006.02050	0.00000
XRSO	2006.02050	0.00000
N
AKYR	2006.17110	-12.48802
ANOP	2006.02050	0.32217
ATRS	2006.02050	-3.39296
GVDS	2006.48600	0.00000
KERY	2006.02050	-1.18801
KITH	2006.02050	-0.02137
KOUN	2006.45860	-22.99194
KRYO	2006.02050	0.51823
MEN1	2006.02050	-0.00049
MET4	2006.02050	-1.02692
NEA1	2006.02050	0.79723
PSAR	2006.44220	-17.12390
RLSO	2006.57630	0.00000
SPR2	2006.55170	0.00000
TUC2	2006.02050	2.39069
VASS	2006.02050	-2.23070
XRSO	2006.02050	-0.62562
U
AKYR	2006.17110	0.00000
ANOP	2006.02050	0.00000
ATRS	2006.02050	0.00000
GVDS	2006.48600	0.00000
KERY	2006.02050	0.00000
KITH	2006.02050	0.00000
KOU

## Read the date of the detected cycle slip and write to file

In [4]:
%%bash

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables
disp=(
    "E"
    "N"
    "U"
)

cd ${dirEXE} || exit 1

for i in $(seq 1 10); do # For each quake

    for j in $(seq 1 2); do # For each nodal plane
        dirIN=${dirEXE}/EQ${i}.${j} # Input Folder
        dirOUT=${dirIN} # Output Folder
        echo EQ${i}.${j} | tee -a CycleSlip.txt

        for g in ${disp[@]}; do # For each component of displacement
            files=("$dirIN"/*."${g}".bayes.out)
            if [ ! -e "${files[0]}" ]; then
                continue
            fi
            echo ${g} | tee -a CycleSlip.txt
    
            # For every .bayes.out file
            for f in "${files[@]}"; do
                sta=$(basename "$f" ."${g}".bayes.out) # Station Code/ Identifier
                gawk -v sta="$sta" -v OFS="\t" '
                    /tau0/ && /=/ {
                        print sta, $NF
                        found = 1
                        exit
                    }
                    END {
                    }
                ' "$f" | tee -a CycleSlip.txt
            done 
        done
    done
done

EQ1.1
E
AKYR	2005.1691
ANOP	2005.1663
ATRS	2005.4018
GVDS	2006.5188
KERY	2005.0486
KITH	2005.0486
KOUN	2005.0486
KRYO	2005.3881
MEN1	2005.0486
MET4	2005.0486
NEA1	2005.0486
PSAR	2005.0486
RLSO	2006.6037
SPR2	2006.5791
TUC2	2005.0486
VASS	2005.0486
XRSO	2005.0486
N
AKYR	2006.2259
ANOP	2005.8508
ATRS	2006.0643
GVDS	2006.7598
KERY	2006.1054
KITH	2005.1581
KOUN	2006.4586
KRYO	2005.8453
MEN1	2005.1225
MET4	2006.0999
NEA1	2005.2320
PSAR	2006.4422
RLSO	2006.8337
SPR2	2006.7515
TUC2	2006.0096
VASS	2006.1027
XRSO	2006.1629
U
AKYR	2005.1691
ANOP	2005.1663
ATRS	2005.4018
GVDS	2006.5188
KERY	2005.0486
KITH	2005.0486
KOUN	2005.0486
KRYO	2005.3881
MEN1	2005.0486
MET4	2005.0486
NEA1	2005.0486
PSAR	2005.0486
RLSO	2006.6037
SPR2	2006.5791
TUC2	2005.0486
VASS	2005.0486
XRSO	2005.0486
EQ1.2
E
AKYR	2005.1691
ANOP	2005.1663
ATRS	2005.4018
GVDS	2006.5188
KERY	2005.0486
KITH	2005.0486
KOUN	2005.0486
KRYO	2005.3881
MEN1	2005.0486
MET4	2005.0486
NEA1	2005.0486
PSAR	2005.0486
RLSO	2006.6037
SPR2	2006.5791
TRIZ	

## Saving Cycle Slip and Respective Coseismic Displacement to One File

In [5]:
%%bash

dirEXE=/mnt/c/Users/Bea/Downloads/Okada/Refining_Thesis/GNSSVerify # Contains Executables
cd ${dirEXE} || exit 1
disp=("E" "N" "U")

for i in $(seq 1 10); do # For each quake
    for j in $(seq 1 2); do # For each nodal plane
        dirIN="${dirEXE}/EQ${i}.${j}" # Input Folder
        dirOUT="${dirIN}" # Output Folder
        echo EQ${i}.${j} | tee -a GNSSData.txt # Save which quake and nodal plane

        for g in "${disp[@]}"; do # For each component of displacement
            files=("$dirIN"/*."${g}".bayes.out) # Contains Cycle Slip Date Detection
            if [ ! -e "${files[0]}" ]; then
                continue
            fi
            echo ${g} | tee -a GNSSData.txt # Save which component
    
            # For every file
            for f in "${files[@]}"; do
                sta=$(basename "$f" ."${g}".bayes.out) # Station Code/ Identifier
                h="$dirIN/$sta.${g}.bayes.p_tau" # Contains Coseismic Slip in mm
                cycle=$(gawk '/tau0/ && /=/ { print $NF; exit }' "$f") # Save Cycle Slip Date in cycle
                coseismic="N/A"
                if [ -n "$cycle" ] && [ -f "$h" ]; then
                    coseismic=$(gawk -v cycle="$cycle" '
                        sprintf("%.5f", $1) == sprintf("%.5f", cycle) {
                            print $3
                            exit
                        }
                    ' "$h")
                fi
                echo -e "${sta}\t${cycle:-N/A}\t${coseismic:-N/A}" | tee -a GNSSData.txt
            done 
        done
    done
done

EQ1.1
E
AKYR	2005.1691	0.00000
ANOP	2005.1663	0.00000
ATRS	2005.4018	0.00000
GVDS	2006.5188	0.00000
KERY	2005.0486	0.00000
KITH	2005.0486	0.00000
KOUN	2005.0486	0.00000
KRYO	2005.3881	0.00000
MEN1	2005.0486	0.00000
MET4	2005.0486	0.00000
NEA1	2005.0486	0.00000
PSAR	2005.0486	0.00000
RLSO	2006.6037	0.00000
SPR2	2006.5791	0.00000
TUC2	2005.0486	0.00000
VASS	2005.0486	0.00000
XRSO	2005.0486	0.00000
N
AKYR	2006.2259	-12.66756
ANOP	2005.8508	2.19523
ATRS	2006.0643	-3.82127
GVDS	2006.7598	1.19544
KERY	2006.1054	-2.69567
KITH	2005.1581	-1.69865
KOUN	2006.4586	-22.99194
KRYO	2005.8453	1.14849
MEN1	2005.1225	-2.53476
MET4	2006.0999	-1.94344
NEA1	2005.2320	-1.44703
PSAR	2006.4422	-17.12390
RLSO	2006.8337	-1.66027
SPR2	2006.7515	1.49773
TUC2	2006.0096	2.50576
VASS	2006.1027	-3.04363
XRSO	2006.1629	-1.62596
U
AKYR	2005.1691	0.00000
ANOP	2005.1663	0.00000
ATRS	2005.4018	0.00000
GVDS	2006.5188	0.00000
KERY	2005.0486	0.00000
KITH	2005.0486	0.00000
KOUN	2005.0486	0.00000
KRYO	2005.3881	0.00000
MEN1	20